# rail-extract: GPU runs on Kaggle

Settings first (right-hand panel): **Accelerator: GPU T4 x2**, **Internet: on**.

Then **Save Version > Save & Run All**. It keeps running if you close the browser.
When it ends, download `rail_extract_output.zip` from the Output tab.

See `docs/KAGGLE.md`.


In [ ]:
# What to run. "overfit" is the first stage (about 15 minutes).
STAGE = "overfit"
REF = "main"  # branch or commit of the public repository
REPO = "https://github.com/cardohso/rail-extract-lora.git"

In [ ]:
import os
import shlex
import subprocess
import sys


def run(cmd, env=None):
    """Run a shell command, stream its output, stop the notebook if it fails."""
    print("$", cmd, flush=True)
    code = subprocess.run(cmd, shell=True, env={**os.environ, **(env or {})}).returncode
    if code:
        sys.exit(f"command failed with exit code {code}: {cmd}")


run("nvidia-smi --query-gpu=name,memory.total --format=csv")
run("pip install -q uv")
run(f"git clone -q {shlex.quote(REPO)} /kaggle/working/repo")
os.chdir("/kaggle/working/repo")
run(f"git checkout -q {shlex.quote(REF)} && git log --oneline -1")
run("uv sync --frozen --group train")
PY = "uv run --frozen --group train python"
GPU = {"CUDA_VISIBLE_DEVICES": "0", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
run(
    f"{PY} -c \"import torch; assert torch.cuda.is_available(), 'no GPU visible'; "
    f'print(torch.__version__, torch.cuda.get_device_name(0))"',
    GPU,
)

In [ ]:
RUNS = "/kaggle/working/runs"
if STAGE == "overfit":
    run(f"{PY} -m rail_extract.train_lora --overfit --out {RUNS}/overfit --ckpt-every 40", GPU)
else:
    sys.exit(f"unknown stage {STAGE}")

In [ ]:
# Everything except the checkpoints goes in one zip.
run(
    "cd /kaggle/working && zip -qr rail_extract_output.zip runs -x '*/training_state.pt' && "
    "ls -la rail_extract_output.zip"
)
run("rm -rf /kaggle/working/repo")  # keep only the outputs in the Output tab